# Football Tactical ILP — Google Colab Workflow

This notebook runs the project pipeline from data preparation through the coach report. It uses interpretable ILP/Popper rules and includes a team selector before data preparation. Select Barcelona or Bayer 04 Leverkusen, then run the cells from top to bottom.

At the coach-review step, download `coach_review.html`, open it in a browser, review each rule, and save the completed `phase6_coach_validation.yaml`. Upload that YAML in the next cell to continue.

Colab sessions are temporary. Download the result ZIP at the end of the run if you need to keep processed data, reports, or logs for a later session.

In [ ]:
# Check the Colab runtime
import sys, platform
print("Python:", platform.python_version())
print("OS:", platform.platform())
!nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null || echo "(GPU is not required for this project)"


## Step 1 — Upload the project ZIP

Upload the current project ZIP. The notebook extracts the selected ZIP and locates the project root automatically.

In [ ]:
# Upload the project ZIP
from google.colab import files
print("Select the project ZIP file...")
uploaded = files.upload()


In [ ]:
# Extract the project ZIP
import zipfile, os, glob, shutil

zip_name = next((f for f in uploaded if f.lower().endswith(".zip")), None)
assert zip_name is not None, "No project ZIP was uploaded"
# Use a dedicated fresh extraction directory so an older upload in this
# Colab session can never win project discovery or leave stale outputs.
EXTRACT_DIR = "/content/football_tactical_ilp_project"
if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)
os.makedirs(EXTRACT_DIR)
with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(EXTRACT_DIR)

candidates = glob.glob(os.path.join(EXTRACT_DIR, "**", "pyproject.toml"), recursive=True)
assert len(candidates) == 1, f"Expected one project root; found {len(candidates)} pyproject.toml files"
PROJECT_DIR = os.path.dirname(candidates[0])
print("Project directory:", PROJECT_DIR)
SOURCE_DIR = os.path.join(PROJECT_DIR, "src")
if not os.path.isfile(os.path.join(SOURCE_DIR, "tactical_ilp", "__init__.py")):
    raise FileNotFoundError(f"The uploaded project archive is missing src/tactical_ilp under {PROJECT_DIR}. Upload the Colab project ZIP, not an output-only archive.")
%cd $PROJECT_DIR
!ls


## Step 2 — Install `uv` and Python dependencies

This step installs the project environment and checks that the source package imports successfully.

In [ ]:
# Install uv and provision the Python ABI required by the pinned OR-Tools wheel
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = f"/root/.local/bin:{os.environ['PATH']}"
!uv --version
!uv python install 3.14


In [ ]:
# Cell 5 — install project dependencies and verify src-layout imports
import os
from pathlib import Path
src_dir = Path(PROJECT_DIR) / "src"
os.environ["PYTHONPATH"] = str(src_dir) + os.pathsep + os.environ.get("PYTHONPATH", "")
!uv sync --extra dev --no-install-project

import sys
sys.path.insert(0, str(src_dir))
import tactical_ilp
import tactical_ilp.data.freeze_frames
print("Package root:", tactical_ilp.__file__)
print("Phase 1 data modules: OK")

## Step 3 — Install SWI-Prolog 9.2.9

The notebook builds SWI-Prolog 9.2.9 from source because the system package may be too old for the `janus_swi` bridge used by Popper. This usually takes several minutes.

In [ ]:
# Install SWI-Prolog build prerequisites
!apt-get update -qq
!apt-get install -y -qq cmake ninja-build libgmp-dev libssl-dev zlib1g-dev libreadline-dev pkg-config build-essential > /dev/null
print("Build prerequisites installed.")


In [ ]:
# Build SWI-Prolog 9.2.9 from source (this may take several minutes)
%cd /content
!rm -rf swipl-devel
!git clone --depth 1 --branch V9.2.9 https://github.com/SWI-Prolog/swipl-devel.git
%cd swipl-devel
!git submodule update --init --depth 1 packages/clib packages/sgml
!mkdir -p build
%cd build
!cmake -G Ninja -DCMAKE_BUILD_TYPE=Release -DSWIPL_PACKAGES=OFF -DCMAKE_INSTALL_PREFIX=/usr/local .. > /content/cmake.log 2>&1
!ninja -j4 > /content/ninja.log 2>&1
!ninja install > /content/install.log 2>&1
!ldconfig
!swipl --version


A successful installation reports `SWI-Prolog version 9.2.9`. If the build fails, inspect `/content/cmake.log`, `/content/ninja.log`, and `/content/install.log` for the failing step.

## Step 4 — Install and check Popper

Popper runs in its own `uv` environment so its Python requirements remain separate from the project environment. Its locked OR-Tools version provides a wheel for Python 3.14, so this cell explicitly selects Python 3.14 instead of inheriting the runtime's newer interpreter.

In [ ]:
# Clone Popper and run its bundled example using Python 3.14
%cd /content
!rm -rf Popper
!git clone --depth 1 https://github.com/logic-and-learning-lab/Popper.git
%cd Popper
!uv run --python 3.14 popper.py examples/iggp-rps-next-score


A successful check prints a Popper `SOLUTION` with precision, recall, and confusion-matrix counts. These example counts come from Popper's bundled example; they are an environment check, not a result from this project.

In [ ]:
# Set the Popper path used by the project
import os
os.environ["POPPER_DIR"] = "/content/Popper"
print("POPPER_DIR =", os.environ["POPPER_DIR"])


## Step 5 — Check the complete runtime

Run the project's environment check. It verifies the external tools and Python components required by the pipeline.

In [ ]:
# Run the project runtime check
%cd $PROJECT_DIR
!POPPER_DIR=/content/Popper uv run --no-sync python scripts/run_phase0_check.py


## Step 5.5 — Choose the team and season

Choose Barcelona (La Liga 2020/21) or Bayer 04 Leverkusen (Bundesliga 2023/24). Click **Use selected team** before running Phase 1. Each choice receives a separate data/cache folder. The pipeline verifies that every match in the selected team-season has 360 frames before continuing.


In [ ]:
# Select a team-season for the complete pipeline
import json
import os
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display
try:
    from statsbombpy import sb
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "statsbombpy"])
    from statsbombpy import sb

TEAM_OPTIONS = {
    "barcelona_2020_21": {
        "label": "Barcelona — La Liga 2020/21",
        "competition_id": 11,
        "season_id": 90,
        "team_name": "Barcelona",
        "match_name_pattern": "Barcelona",
    },
    "leverkusen_2023_24": {
        "label": "Bayer 04 Leverkusen — Bundesliga 2023/24",
        "competition_id": 9,
        "season_id": 281,
        "team_name": "Bayer 04 Leverkusen",
        "match_name_pattern": "Leverkusen",
    },
}
team_selector = widgets.Dropdown(
    options=[(v["label"], k) for k, v in TEAM_OPTIONS.items()],
    value="barcelona_2020_21",
    description="Team:",
    layout=widgets.Layout(width="560px"),
)
apply_team_button = widgets.Button(
    description="Use selected team", button_style="primary", icon="check"
)
team_selection_output = widgets.Output()
TEAM_SELECTION_CONFIGURED = False
DATA_DIR = None


def _apply_team_selection(_):
    global TEAM_SELECTION_CONFIGURED, DATA_DIR
    TEAM_SELECTION_CONFIGURED = False
    DATA_DIR = None
    spec = TEAM_OPTIONS[team_selector.value]
    with team_selection_output:
        team_selection_output.clear_output()
        matches = sb.matches(
            competition_id=spec["competition_id"], season_id=spec["season_id"]
        )
        pattern = spec["match_name_pattern"]
        home = matches["home_team"].astype(str)
        away = matches["away_team"].astype(str)
        mask = home.str.contains(pattern, case=False, regex=False) | away.str.contains(
            pattern, case=False, regex=False
        )
        selected_matches = matches.loc[mask].sort_values("match_id")
        if selected_matches.empty:
            names = sorted(set(home) | set(away))
            candidates = [n for n in names if pattern.casefold() in n.casefold()]
            raise RuntimeError(
                f"No matches found for {spec['team_name']}; matching names: {candidates}"
            )

        first_match = selected_matches.iloc[0]
        first_match_id = int(first_match["match_id"])
        resolved_name = next(
            (name for name in (first_match["home_team"], first_match["away_team"])
             if pattern.casefold() in str(name).casefold()),
            None,
        )
        if resolved_name is None:
            raise RuntimeError(f"Could not identify team in match {first_match_id}.")

        events = sb.events(match_id=first_match_id, fmt="dataframe")
        team_col = next(
            (col for col in ("team_name", "team") if col in events.columns), None
        )
        if team_col is None or "team_id" not in events.columns:
            raise RuntimeError(
                f"Missing team/team_id event columns. Available: {list(events.columns)}"
            )
        team_rows = events.loc[
            events[team_col].astype(str).str.casefold() == str(resolved_name).casefold(),
            "team_id",
        ].dropna()
        if team_rows.empty:
            raise RuntimeError(
                f"Could not resolve team id for {resolved_name!r} in match {first_match_id}."
            )
        team_id = int(team_rows.iloc[0])

        os.environ["TACTICAL_ILP_COMPETITION_ID"] = str(spec["competition_id"])
        os.environ["TACTICAL_ILP_SEASON_ID"] = str(spec["season_id"])
        os.environ["TACTICAL_ILP_TEAM_NAME"] = str(resolved_name)
        os.environ["TACTICAL_ILP_TEAM_ID"] = str(team_id)
        DATA_DIR = Path(PROJECT_DIR) / "data" / "team_runs" / team_selector.value
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        os.environ["TACTICAL_ILP_DATA"] = str(DATA_DIR)
        manifest = {
            "team_name": str(resolved_name),
            "competition_id": spec["competition_id"],
            "season_id": spec["season_id"],
            "match_ids": [int(mid) for mid in selected_matches["match_id"]],
            "available_match_count": len(selected_matches),
            "statsbomb_team_id": team_id,
        }
        (DATA_DIR / "selected_dataset.json").write_text(
            json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
        )
        TEAM_SELECTION_CONFIGURED = True
        print(f"Selected: {spec['label']}")
        print(
            f"Available matches: {len(selected_matches)} | StatsBomb name: "
            f"{resolved_name} | team id: {team_id}"
        )
        if team_selector.value == "barcelona_2020_21" and len(selected_matches) < 38:
            print(
                f"StatsBomb Open Data returns {len(selected_matches)} Barcelona "
                "matches here, not all 38 league fixtures; the pipeline will use "
                "the available matches only."
            )
        print(f"Isolated data/cache directory: {DATA_DIR}")
        print("Selection applied. Continue with Phase 1.")


apply_team_button.on_click(_apply_team_selection)
display(widgets.VBox([team_selector, apply_team_button, team_selection_output]))


## Step 6 — Run the pipeline

Run phases 1–3 to prepare the data and extract features. Choose a team in the next cell. Each team has its own data/cache folder under data/team_runs, so its outputs can be reused from a prior result ZIP without mixing teams. Missing outputs are generated.

In [ ]:
# Phase 1: prepare the selected team
assert TEAM_SELECTION_CONFIGURED and DATA_DIR is not None, "Choose a team and click Use selected team first."
!uv run --no-sync python scripts/run_phase1.py


In [ ]:
# Phase 2: build sequences and labels
!uv run --no-sync python scripts/run_phase2.py


In [ ]:
# Phase 3: extract features
!uv run --no-sync python scripts/run_phase3.py


## Step 6.5 — Phase 4: tactical facts and Prolog background knowledge

Phase 4 copies the source background-knowledge file to `data/prolog/bk.pl` when needed, derives tactical facts from the 360 frames, and checks the Prolog rules. It reports predicates with no matches as warnings; a Prolog syntax error stops the pipeline.

In [ ]:
# Run Phase 4, then verify its outputs under the selected team data directory
%cd $PROJECT_DIR

assert TEAM_SELECTION_CONFIGURED and DATA_DIR is not None, "Select a team first."

# Use the notebook shell so run_logged() output is streamed into this cell
!uv run --no-sync python -u scripts/run_phase4.py

from pathlib import Path
required = [DATA_DIR / "prolog" / "facts_tactical.pl", DATA_DIR / "prolog" / "bk.pl"]
missing = [str(path) for path in required if not path.is_file()]
assert not missing, f"Phase 4 outputs were not created: {missing}. Check the Phase 4 run log."
print("Phase 4 outputs verified:", ", ".join(str(path) for path in required))


## Optional Arm 0 — reference: `full_tactical` Popper

This historical full-vocabulary Popper route is retained as a secondary reference. It is not part of the primary comparison and is skipped by default to keep the main run focused. Set `RUN_ARM0_REFERENCE = True` in the code cell below if you want to rebuild its training output; the primary Arm 1–3 comparison does not require it.

In [ ]:
%cd $PROJECT_DIR
import os
import subprocess
from pathlib import Path
RUN_ARM0_REFERENCE = bool(globals().get("RUN_ARM0_REFERENCE", False))
if RUN_ARM0_REFERENCE:
 env = os.environ.copy()
 env["POPPER_DIR"] = "/content/Popper"
 env["TACTICAL_ILP_DATA"] = str(Path(DATA_DIR).resolve())
 result = subprocess.run(["uv", "run", "--no-sync", "python", "-u", "scripts/run_phase5.py"], cwd=str(PROJECT_DIR), env=env, text=True)
 result.check_returncode()
 p = DATA_DIR / "reports" / "phase5_popper_output.txt"
 if not p.is_file(): raise RuntimeError("The full Popper log for Arm 0 was not created.")
 print(f"\nFull log: {p} ({p.stat().st_size:,} bytes)")
else:
 print("Arm 0 reference skipped. Set RUN_ARM0_REFERENCE = True here to run it; the primary comparison does not require Arm 0.")

### Optional Arm 0 CV — `full_tactical` reference

Run this only when you want a fresh match-grouped CV result for the secondary reference. It is not required for the primary comparison cell; that cell reads Arm 0 metrics only as an optional reference when this report exists.

In [ ]:
%cd $PROJECT_DIR
import os
import subprocess
from pathlib import Path
RUN_ARM0_REFERENCE = bool(globals().get("RUN_ARM0_REFERENCE", False))
if RUN_ARM0_REFERENCE:
 env = os.environ.copy()
 env["POPPER_DIR"] = "/content/Popper"
 env["TACTICAL_ILP_DATA"] = str(Path(DATA_DIR).resolve())
 result = subprocess.run(["uv", "run", "--no-sync", "python", "-u", "scripts/run_phase5_cv.py", "--folds", "3", "--seed", "202021", "--timeout", "600"], cwd=str(PROJECT_DIR), env=env, text=True)
 result.check_returncode()
 p = DATA_DIR / "reports" / "phase5_cross_validation.json"
 if not p.is_file(): raise RuntimeError("The Arm 0 evaluation report was not created.")
 print("Structured report:", p)
 d = DATA_DIR / "reports" / "phase5_cross_validation"
 if d.is_dir():
  for f in sorted(d.glob("*")):
   if f.is_file(): print(f"\n--- {f} ---\n" + f.read_text(encoding="utf-8", errors="replace"))
else:
 print("Arm 0 CV skipped. Set RUN_ARM0_REFERENCE = True in its code cell to run it; the primary comparison does not require Arm 0.")

### Optional — rescore CV with distinct-sequence counting
If you have a saved grouped-CV report from an earlier run, this re-evaluates its stored fold rules using the corrected once-per-sequence scoring and preserves original metrics. It does not rerun Popper. The new report is written beside the original. Run after the grouped CV cell and while its fold task folders are available.

In [ ]:
from pathlib import Path
import subprocess, sys
report_path = DATA_DIR / "reports" / "phase5_cross_validation.json"
rescored_path = DATA_DIR / "reports" / "phase5_cross_validation_rescored.json"
task_pattern = str(DATA_DIR / "prolog" / "popper_cv_fold_{fold:02d}")
if not report_path.is_file():
 print("Arm 0 CV report is absent; optional rescoring skipped.")
else:
 subprocess.run([sys.executable, "-u", "scripts/rescore_cv.py", "oof", "--report", str(report_path), "--task-pattern", task_pattern, "--out", str(rescored_path)], check=True)
 print("Rescored report:", rescored_path)

### Arm 3, step 1 of 3 — Mine patterns and prepare hybrid inputs

Sequential patterns are mined from training matches only. Popper is not run in this cell. The prepared-input summary, including mining settings, match splits, and any fallback, is saved as `data/team_runs/<selected-team>/reports/phase5_sequential_prepared.json`.

In [ ]:
%cd $PROJECT_DIR
!uv run --no-sync python -u scripts/run_phase5_sequential.py --prepare-only --max-gap 2 --min-support 5 --min-target-support 5 --min-growth-rate 1.5 --max-patterns 8 --max-candidates 50000 --min-events 2 --max-events 3 --exclude-suffix-actions 1
import json
from pathlib import Path
p = DATA_DIR / "reports" / "phase5_sequential_prepared.json"
if not p.is_file(): raise RuntimeError("The Arm 3 pattern-mining report was not created.")
print(json.dumps(json.loads(p.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))


### Arm 3, step 2 of 3 — Train the hybrid Popper model

Popper uses the revised active tactical vocabulary plus the selected sequential pattern atoms. The combined bias has 34 base predicates and up to 8 additional patterns (42 total). This is the hybrid method, not the miner-only arm. The search log and learned-rule summary are saved under the selected team's reports folder.

In [ ]:
%cd $PROJECT_DIR
import os, json
from pathlib import Path
os.environ["POPPER_DIR"] = "/content/Popper"
!POPPER_DIR=/content/Popper uv run --no-sync python -u scripts/run_phase5_sequential.py --popper-only --timeout 900
for p in [DATA_DIR / "reports" / "phase5_sequential_training_popper_output.txt", DATA_DIR / "reports" / "phase5_sequential_training.json"]:
 if p.is_file():
  print(f"\n--- {p} ---")
  text = p.read_text(encoding="utf-8", errors="replace")
  if p.suffix == ".json": text = json.dumps(json.loads(text), ensure_ascii=False, indent=2)
  print(text)


### Arm 3, step 3 of 3 — Evaluate hybrid miner + Popper

For each fold, patterns are mined from that fold's training matches, Popper is run, and the rules are evaluated on unseen matches. This same CV run also scores Arm 1 (miner-only) using the fold-local pattern union, with no Popper. Results are saved in `phase5_sequential_cv.json/.md`.

In [ ]:
%cd $PROJECT_DIR
import os
from pathlib import Path
from IPython.display import Markdown, display
os.environ["POPPER_DIR"] = "/content/Popper"
!POPPER_DIR=/content/Popper uv run --no-sync python -u scripts/run_phase5_sequential_cv.py --folds 3 --seed 202021 --timeout 600 --max-gap 2 --min-support 5 --min-target-support 5 --min-growth-rate 1.5 --max-patterns 8 --max-candidates 50000 --min-events 2 --max-events 3 --exclude-suffix-actions 1
p = DATA_DIR / "reports" / "phase5_sequential_cv.md"
if not p.is_file(): raise RuntimeError("The Arm 3 evaluation report was not created.")
display(Markdown(p.read_text(encoding="utf-8")))
d = DATA_DIR / "reports" / "phase5_sequential_cv"
if d.is_dir():
 for f in sorted(d.glob("*")):
  if f.is_file(): print(f"\n--- {f} ---\n" + f.read_text(encoding="utf-8", errors="replace"))


### Arm 1 — Miner-only baseline

This method predicts a negative outcome when any compressed fold-local pattern offered to the hybrid Popper matches the sequence (OR rule). It has no Popper and does not combine patterns with the base tactical predicates. This makes Arm 1 a direct ablation of Popper rule search in Arm 3. The score is computed automatically inside the Arm 3 grouped CV from that fold's training-only patterns and exactly the same held-out matches; there is no separate evaluation run. The results are in `miner_only_out_of_fold_metrics` and each fold's `miner_only_test_metrics` in `phase5_sequential_cv.json`.

### Arm 2 — Classical baseline: shallow decision tree

This CART tree uses the same 34 sequence-level Boolean tactical features and context-matched training examples, with the same three match-grouped folds. It is capped at depth 3 with at least 10 training examples per leaf. The report contains pooled out-of-fold metrics and fold-specific trees.

In [ ]:
%cd $PROJECT_DIR
import os, subprocess
env = os.environ.copy()
env["TACTICAL_ILP_DATA"] = str(DATA_DIR.resolve())
result = subprocess.run(["uv", "run", "--no-sync", "python", "-u", "scripts/run_phase5_decision_tree.py", "--folds", "3", "--seed", "202021"], cwd=str(PROJECT_DIR), env=env, text=True)
result.check_returncode()
import json
from pathlib import Path
p = DATA_DIR / "reports" / "phase5_decision_tree_cv.json"
if not p.is_file(): raise RuntimeError("The decision-tree baseline report was not created.")
report = json.loads(p.read_text(encoding="utf-8"))
print(json.dumps(report["out_of_fold_metrics"], ensure_ascii=False, indent=2))
for fold in report["fold_results"]:
 print("\n--- Fold", fold["fold"], "decision tree ---")
 print(fold["readable_tree"])

In [ ]:
# Compare the three primary methods: Arm 1 miner-only, Arm 2 CART, Arm 3 hybrid
import json
import os
import subprocess
from pathlib import Path
env = os.environ.copy()
env["TACTICAL_ILP_DATA"] = str(Path(DATA_DIR).resolve())
subprocess.run(["uv", "run", "--no-sync", "python", "scripts/compare_phase5_methods.py"], cwd=str(PROJECT_DIR), env=env, check=True)
p = DATA_DIR / "reports" / "phase5_method_comparison.md"
if not p.is_file(): raise RuntimeError("The three-method comparison report was not created.")
report = json.loads((DATA_DIR / "reports" / "phase5_method_comparison.json").read_text(encoding="utf-8"))
print(p.read_text(encoding="utf-8"))
if not report.get("comparison_ready"):
 print("Comparison is not ready. Re-run the updated Arm 3 CV and Arm 2 decision-tree CV cells; check that both reports use the current protocol and shared match folds.")

## Step 7 — Phase 6: prepare the coach review

The default method is `hybrid_sequential`. Phase 6 reads artifacts for that method. To review the `full_tactical` baseline instead, set `TACTICAL_ILP_COACH_METHOD=full_tactical` and run that arm's cross-validation first.

In [ ]:
# Create the coach-review form for the hybrid arm
import os
os.environ["TACTICAL_ILP_COACH_METHOD"] = "hybrid_sequential"
!uv run --no-sync python scripts/run_phase6.py

Download and open `coach_review.html` in a browser. Approve or reject each rule and optionally add notes. The form saves the decisions as `phase6_coach_validation.yaml`; upload that file in the next cell. The HTML file is the interactive review form, while the YAML file stores the coach's decisions and notes.

In [ ]:
# Generate the coach-review form in the selected team report folder
import subprocess
from google.colab import files
yaml_path = DATA_DIR / "reports" / "phase6_coach_validation.yaml"
html_path = DATA_DIR / "reports" / "coach_review.html"
subprocess.run(["uv", "run", "--no-sync", "python", "scripts/generate_coach_review_ui.py", "--yaml", str(yaml_path), "--html", str(html_path)], check=True)
files.download(str(html_path))


In [ ]:
# Upload and validate the reviewed YAML
from google.colab import files
from pathlib import Path
import subprocess
uploaded_review = files.upload()
yaml_candidates = [name for name in uploaded_review if Path(name).suffix.lower() in {".yaml", ".yml"}]
review_name = next((name for name in yaml_candidates if "phase6_coach_validation" in Path(name).name), None)
if review_name is None and len(yaml_candidates) == 1:
    review_name = yaml_candidates[0]
if review_name is None:
    raise ValueError(f"Upload one reviewed coach YAML file; received: {list(uploaded_review)}")
review_path = DATA_DIR / "reports" / "phase6_coach_validation.yaml"
review_path.parent.mkdir(parents=True, exist_ok=True)
review_path.write_bytes(uploaded_review[review_name])
check = subprocess.run(["uv", "run", "python", "-c", "from tactical_ilp.evaluation.coach_validation import validate_coach_file; ok, problems = validate_coach_file(); print(chr(10).join(problems)); raise SystemExit(0 if ok else 1)"], text=True, capture_output=True)
print(check.stdout)
if check.returncode:
    print(check.stderr)
    raise ValueError("Coach review is incomplete or invalid. Edit the YAML and rerun this cell.")
print("Coach review is valid.")

## Step 8 — Phase 7: build the final coach report

Phase 7 reads the reviewed YAML and builds the self-contained `data/team_runs/<selected-team>/reports/coach_report.html`. The report presents approved and rejected rules, train/holdout metrics, supporting evidence, and charts. It also uses `data/team_runs/<selected-team>/reports/phase5_weighted_rules.json` for the statistical weights and combined risk estimates when available. The HTML report can be opened offline.

In [ ]:
# Generate the coach report from the reviewed Phase 6 YAML
%cd $PROJECT_DIR
import os
import subprocess
from pathlib import Path
assert TEAM_SELECTION_CONFIGURED and DATA_DIR is not None, "Select a team first."
yaml_path = DATA_DIR / "reports" / "phase6_coach_validation.yaml"
assert yaml_path.is_file(), f"Reviewed coach YAML not found: {yaml_path}; complete Phase 6 review first."
env = os.environ.copy()
env["TACTICAL_ILP_DATA"] = str(Path(DATA_DIR).resolve())
result = subprocess.run(["uv", "run", "--no-sync", "python", "scripts/run_phase7.py"], cwd=str(PROJECT_DIR), env=env, text=True, capture_output=True)
print(result.stdout)
if result.stderr: print(result.stderr)
result.check_returncode()
report_html = Path(DATA_DIR) / "reports" / "coach_report.html"
if not report_html.is_file(): raise FileNotFoundError(f"Phase 7 exited successfully but did not create the expected report: {report_html}")
print(f"Coach report created: {report_html} ({report_html.stat().st_size:,} bytes)")

In [ ]:
# Confirm the generated coach report
from pathlib import Path
report_html = Path(DATA_DIR) / "reports" / "coach_report.html"
if not report_html.is_file():
 reports_dir = Path(DATA_DIR) / "reports"
 candidates = sorted(str(path) for path in reports_dir.rglob("coach_report.html")) if reports_dir.is_dir() else []
 raise FileNotFoundError(f"Coach report not found at {report_html}. Run the Phase 7 generation cell above and check its output. Other matching reports: {candidates}")
print(f"Coach report: {report_html} ({report_html.stat().st_size:,} bytes)")

In [ ]:
from google.colab import files
files.download(str(DATA_DIR / "reports" / "coach_report.html"))


## Step 9 — Download the complete result package

The result ZIP contains the coach report, review form and reviewed YAML when present, phase logs and evaluation summaries, processed data, Prolog facts, source code, tests, and documentation. It writes directory entries as well as files, so empty folders are preserved. The notebook creates `/content/football-tactical-ilp-result.zip` and downloads it.

### Descriptive analysis of terminal actions

This report counts terminal-action types among negative-outcome sequences covered by the learned rules. Terminal actions are analyzed after rule discovery and are not added to Popper's input vocabulary. Action-type categories are exclusive; pass labels may overlap.

In [ ]:
%cd $PROJECT_DIR
!uv run --no-sync python scripts/run_phase5_terminal_action_analysis.py
from pathlib import Path
from IPython.display import Markdown, display
report_path = DATA_DIR / "reports" / "phase5_terminal_action_analysis.md"
display(Markdown(report_path.read_text(encoding="utf-8")))

In [ ]:
# Package and download outputs for the selected team
%cd $PROJECT_DIR
import os, shutil, zipfile
from pathlib import Path
from google.colab import files

assert TEAM_SELECTION_CONFIGURED and DATA_DIR is not None, "Choose a team and click Use selected team first."
result_dir = Path("/content/result_package")
if result_dir.exists():
    shutil.rmtree(result_dir)
result_dir.mkdir(parents=True)
for name in ["report.md", "README.md", "DOCs", "history_docs", "scripts", "src", "tests",
             "pyproject.toml", "uv.lock", "football_tactical_ilp_colab_setup.ipynb"]:
    source = Path(PROJECT_DIR) / name
    if source.is_dir():
        shutil.copytree(source, result_dir / name)
    elif source.exists():
        shutil.copy2(source, result_dir / name)
(result_dir / "data" / "team_runs").mkdir(parents=True, exist_ok=True)
shutil.copytree(DATA_DIR, result_dir / "data" / "team_runs" / team_selector.value)
zip_path = Path("/content/football-tactical-ilp-result.zip")
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for item in sorted(result_dir.rglob("*")):
        archive_name = item.relative_to(result_dir).as_posix()
        if item.is_dir():
            archive.write(item, archive_name.rstrip("/") + "/")
        else:
            archive.write(item, archive_name)
files.download(str(zip_path))
print(f"Downloaded project results for {team_selector.label}: {zip_path}")


## Continue in a later Colab session

Colab storage is temporary. Upload the result ZIP in Step 1 of a new session to reuse processed data and reports. Phases 1–3 reuse complete cached outputs when available. To rebuild them explicitly, add `--force` to the relevant phase command. Phase 7 creates the coach report, and the final cell downloads the complete result ZIP.